# Multi-Head Stance Model — Turn-Level Analysis (v3, 6 models)

Adapts the chunk-level multi-head DeBERTa analysis to **speaker turns** as the unit of analysis,
now across **six LLMs**: DeepSeek V3, Gemini 2.5 Flash, GPT-4o Mini, Llama 3.3, Mistral Large,
Qwen 2.5 72B.

## Structure
0. **Health check** — verify all 6 prediction files are complete, clean, and mergeable
1. **Data loading** — binary `split` target + graded companions (`label_entropy`,
   `max_ordinal_gap`, `mean_score`, `score_std`, `mean_pair_d`)
2. **Dictionary baseline** — hawk/dove word count → predict disagreement (AUC floor)
3. **TF-IDF + SHAP** — statistical text features → predict disagreement
4. **Multi-Head DeBERTa** — shared encoder + one linear head per LLM, trained jointly
5. **Per-head metrics** — accuracy / macro-F1 + confusion matrices
6. **Linear probe + base-vs-finetuned** — frozen pretrained vs fine-tuned CLS AUC; graded
   companions (entropy + score-dispersion regressions)
7. **Swap analysis** — encoding vs decision-rule decomposition (shared encoder, all heads)
8. **Gradient attribution** — which tokens drive each head's prediction?
9. **Head weight geometry** — cosine similarity + PCA of the learned linear projections

**No ModernBERT.** The truncation diagnostic (`15_truncation_diagnostic.ipynb`) showed split
rates plateau across the 384–512 / 512–768 / 768–1024 token buckets with no discontinuity at the
512-token cutoff — truncation is a length confound, not a measurement artifact.

**Upload** these to `/content/` before running:
`turn_predictions_{deepseekv3,gemini25flash,gpt-4o,llama33,mistrallarge_or,qwen25_72b}.csv`

**Runtime:** ~30–40 min on T4 GPU for DeBERTa training.

In [ ]:
!pip install -q transformers accelerate scikit-learn shap scipy matplotlib seaborn

In [ ]:
import os, re, random
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DebertaV2Model
from scipy.stats import entropy, spearmanr
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay,
                             accuracy_score, f1_score, r2_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Set DATA_DIR to wherever the 6 CSVs live: '/content/' on Colab, '../output/stance/' locally.
DATA_DIR = "/content/" if os.path.exists("/content") else "../output/stance/"

MODELS = {
    "deepseekv3":      "DeepSeek V3",
    "gemini25flash":   "Gemini 2.5 Flash",
    "gpt-4o":          "GPT-4o Mini",
    "llama33":         "Llama 3.3",
    "mistrallarge_or": "Mistral Large",
    "qwen25_72b":      "Qwen 2.5 72B",
}

LABEL2IDX  = {"dovish":0,"mostly dovish":1,"neutral":2,"mostly hawkish":3,"hawkish":4}
IDX2LABEL  = {v:k for k,v in LABEL2IDX.items()}
LABEL_ORDER = list(LABEL2IDX.keys())
STANCED    = {"dovish","mostly dovish","mostly hawkish","hawkish"}
SCORE_MAP  = {"dovish":-2,"mostly dovish":-1,"neutral":0,"mostly hawkish":1,"hawkish":2}
MODEL_NAME = "microsoft/deberta-v3-base"

def path_for(key):
    return f"{DATA_DIR}turn_predictions_{key}.csv"

## 0. Health Check

Verify every model file exists, is fully labeled, has zero `parse_error` rows, and that all
six share an identical turn set and text. **Hard-fails** if any file is incomplete — an
unfinished scoring run must not silently shrink the analysis dataset.

In [ ]:
raw = {}
rows = []
for key in MODELS:
    p = path_for(key)
    assert os.path.exists(p), f"MISSING: {p}"
    df = pd.read_csv(p)
    raw[key] = df
    n        = len(df)
    labeled  = df["label"].notna().sum()
    pe       = int((df["label"] == "parse_error").sum())
    valid    = df["label"].isin(LABEL2IDX).sum()
    neutral  = (df["label"] == "neutral").mean()
    rows.append({"model":MODELS[key], "rows":n, "labeled":int(labeled),
                 "parse_error":pe, "valid_labels":int(valid),
                 "neutral_rate":f"{neutral:.1%}"})

health = pd.DataFrame(rows).set_index("model")
print(health.to_string())

# Hard gates
for key, df in raw.items():
    incomplete = df["label"].notna().sum() < len(df)
    has_pe     = (df["label"] == "parse_error").any()
    assert not incomplete, f"{key}: incomplete ({df['label'].notna().sum()}/{len(df)}) — finish scoring first"
    assert not has_pe,     f"{key}: contains parse_error rows — re-run to repair before analysis"
print("\nAll 6 files complete with 0 parse_errors.")

In [ ]:
# Merge integrity: identical turn_uid set and identical text across all 6 files.
ref_key = list(MODELS)[0]
ref     = raw[ref_key][["turn_uid", "text"]].drop_duplicates("turn_uid").set_index("turn_uid")
for key in MODELS:
    s = set(raw[key]["turn_uid"])
    assert s == set(ref.index), f"{key}: turn_uid set differs from {ref_key}"
    merged_txt = raw[key][["turn_uid","text"]].drop_duplicates("turn_uid").set_index("turn_uid")
    mism = (merged_txt["text"] != ref["text"]).sum()
    assert mism == 0, f"{key}: {mism} turn_uid(s) have different text than {ref_key}"
print(f"Merge integrity OK — all 6 models share {len(ref):,} identical turns.")

# Calibration-bias ordering (most→least neutral)
order = (health["neutral_rate"].str.rstrip('%').astype(float).sort_values(ascending=False))
print("\nNeutral-rate ordering (calibration conservatism):")
for m, v in order.items():
    print(f"  {m:18s} {v:.1f}%")

## 1. Data Loading

`split = True` when between 1 and 5 of the 6 models classify a turn as directional while the
rest call it neutral — partial disagreement on whether the turn takes a stance at all.
With six models the binary base rate is high, so we also compute continuous companions:

- `label_entropy` — entropy over the 5-class **nominal** label distribution (ignores ordering)
- `max_ordinal_gap` — range (max−min) on the dovish→hawkish ordinal scale
- `mean_score` — average hawkishness (−2…+2); this is the **stance signal**, not disagreement
- `score_std` — cross-sectional **population SD** of the 6 ordinal scores; the direct analogue
  of forecast dispersion in the Malta synthetic-traders paper. Unlike entropy it respects the
  ordering, so it captures *how far apart* the models are, not just that they differ.
- `mean_pair_d` — mean absolute pairwise distance (ordinal-honest sibling of `score_std`:
  "two randomly chosen LLMs differ by X categories on average")

In [ ]:
dfs = {}
for key in MODELS:
    df = raw[key]
    df = df[df["label"].isin(LABEL2IDX)].copy()
    dfs[key] = df
    print(f"{key}: {len(df)} turns")

base_cols  = ["turn_uid","bank","date","doc_type","speaker","speaker_role","turn_idx","text"]
first_key  = list(MODELS.keys())[0]
turns_wide = dfs[first_key][base_cols].copy()
loaded     = []

for key, df in dfs.items():
    sub = df[["turn_uid","label"]].rename(columns={"label":f"label_{key}"})
    turns_wide = turns_wide.merge(sub, on="turn_uid", how="inner")
    turns_wide[f"score_{key}"]   = turns_wide[f"label_{key}"].map(SCORE_MAP)
    turns_wide[f"stanced_{key}"] = turns_wide[f"label_{key}"].isin(STANCED).astype(int)
    loaded.append(key)

n_models = len(loaded)
turns_wide["n_stanced"] = turns_wide[[f"stanced_{k}" for k in loaded]].sum(axis=1)
turns_wide["split"]     = turns_wide["n_stanced"].between(1, n_models-1)

def label_entropy_norm(row):
    labels = [row[f"label_{k}"] for k in loaded]
    counts = [labels.count(l) for l in LABEL_ORDER]
    return entropy(counts, base=5)  # 5 label classes -> normalized to [0,1]

def max_ordinal_gap(row):
    ords = [LABEL2IDX[row[f"label_{k}"]] for k in loaded]
    return max(ords) - min(ords)

turns_wide["label_entropy"]   = turns_wide.apply(label_entropy_norm, axis=1)
turns_wide["max_ordinal_gap"] = turns_wide.apply(max_ordinal_gap, axis=1)

# Cross-sectional dispersion of the 6 ordinal scores (Malta synthetic-traders analogue):
#   mean_score = average hawkishness (the stance signal); score_std = disagreement magnitude.
# Population SD (ddof=0): the 6 raters are the whole panel, not a sample.
score_cols = [f"score_{k}" for k in loaded]
turns_wide["mean_score"]  = turns_wide[score_cols].mean(axis=1)
turns_wide["score_std"]   = turns_wide[score_cols].std(axis=1, ddof=0)
turns_wide["mean_pair_d"] = turns_wide[score_cols].apply(
    lambda r: np.mean([abs(a - b) for a, b in combinations(r.values, 2)]), axis=1)

rng  = np.random.RandomState(SEED)
uids = turns_wide["turn_uid"].unique()
rng.shuffle(uids)
n    = len(uids)
n_tr = int(0.70*n); n_va = int(0.15*n)
split_map = {uid:"train" for uid in uids[:n_tr]}
split_map.update({uid:"val"  for uid in uids[n_tr:n_tr+n_va]})
split_map.update({uid:"test" for uid in uids[n_tr+n_va:]})
turns_wide["split_set"] = turns_wide["turn_uid"].map(split_map)

print(f"\nTotal turns : {len(turns_wide)}")
print(f"Split turns : {turns_wide['split'].sum()} ({turns_wide['split'].mean():.1%})")
print(f"label_entropy   mean={turns_wide['label_entropy'].mean():.3f}  max={turns_wide['label_entropy'].max():.3f}")
print(f"max_ordinal_gap mean={turns_wide['max_ordinal_gap'].mean():.2f}  max={turns_wide['max_ordinal_gap'].max()}")
print(f"score_std       mean={turns_wide['score_std'].mean():.3f}  max={turns_wide['score_std'].max():.3f}")
print(f"mean_pair_d     mean={turns_wide['mean_pair_d'].mean():.3f}  max={turns_wide['mean_pair_d'].max():.3f}")
print(f"mean_score      mean={turns_wide['mean_score'].mean():+.3f}  (overall hawkishness signal)")
print(f"Train: {(turns_wide['split_set']=='train').sum()}  "
      f"Val: {(turns_wide['split_set']=='val').sum()}  "
      f"Test: {(turns_wide['split_set']=='test').sum()}")

## 2. Baseline 1 — Dictionary (AUC Floor)

Count hawk/dove words per turn; net score predicts disagreement. The simplest possible text
feature — if TF-IDF and DeBERTa cannot beat this, the complex methods add nothing.

In [ ]:
HAWK_WORDS = {
    "hike","hikes","hiking","tighten","tightening","tightened",
    "raise","raises","raising","restrictive","restriction",
    "inflation","inflationary","overshoot","overheating",
    "hawkish","normalisation","normalization","unwind",
}
DOVE_WORDS = {
    "cut","cuts","cutting","accommodation","accommodative",
    "stimulus","easing","ease","support","lower","lowering",
    "dovish","expansionary","unconventional","qe","purchase",
    "below","undershoot",
}

def dict_score(text):
    tokens = re.findall(r"[a-z]+", str(text).lower())
    h = sum(1 for t in tokens if t in HAWK_WORDS)
    d = sum(1 for t in tokens if t in DOVE_WORDS)
    return h - d

turns_wide["dict_score"] = turns_wide["text"].apply(dict_score)

tr_mask = turns_wide["split_set"] == "train"
te_mask = turns_wide["split_set"] == "test"
y_tr = turns_wide.loc[tr_mask,"split"].astype(int).values
y_te = turns_wide.loc[te_mask,"split"].astype(int).values

X_dict_tr = turns_wide.loc[tr_mask,"dict_score"].values.reshape(-1,1)
X_dict_te = turns_wide.loc[te_mask,"dict_score"].values.reshape(-1,1)

lr_dict  = LogisticRegression(class_weight="balanced")
lr_dict.fit(X_dict_tr, y_tr)
auc_dict = roc_auc_score(y_te, lr_dict.predict_proba(X_dict_te)[:,1])
print(f"Dictionary AUC: {auc_dict:.3f}")

## 3. Baseline 2 — TF-IDF + SHAP

Statistical text features weighted by corpus frequency. SHAP shows which words drive
prediction of split vs. consensus. Compare these top words against the DeBERTa gradient
attribution later — shared vocabulary means the finding is robust to method.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import shap

train_texts = turns_wide.loc[tr_mask,"text"].tolist()
test_texts  = turns_wide.loc[te_mask,"text"].tolist()

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                        min_df=3, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(train_texts)
X_te_tfidf = tfidf.transform(test_texts)

lr_tfidf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
lr_tfidf.fit(X_tr_tfidf, y_tr)
auc_tfidf = roc_auc_score(y_te, lr_tfidf.predict_proba(X_te_tfidf)[:,1])
print(f"TF-IDF AUC : {auc_tfidf:.3f}")
print(f"Dictionary : {auc_dict:.3f}")

explainer   = shap.LinearExplainer(lr_tfidf, X_tr_tfidf,
                                    feature_perturbation="interventional")
shap_vals   = explainer.shap_values(X_te_tfidf)
feat_names  = tfidf.get_feature_names_out()
mean_shap   = np.abs(shap_vals).mean(axis=0)
top_idx     = np.argsort(mean_shap)[::-1][:25]

fig, ax = plt.subplots(figsize=(10,6))
ax.barh(range(25), mean_shap[top_idx][::-1], color="#4C72B0")
ax.set_yticks(range(25))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("TF-IDF Top 25 Features Predicting LLM Disagreement (6 models)")
plt.tight_layout()
plt.savefig("tfidf_shap_disagreement_turns.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Multi-Head DeBERTa

```
turn text -> DeBERTa-v3-base -> [CLS] 768-dim
                                     |
          one Linear(768,5) head per model (6 heads), trained jointly
```

All heads trained simultaneously; the shared encoder satisfies all six labeling functions.

**Note:** Turns are truncated to 512 tokens. Gradient attribution covers the first ~400 words.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiHeadDataset(Dataset):
    def __init__(self, df, model_keys, max_length=512):
        self.df = df.reset_index(drop=True)
        self.model_keys = model_keys
        self.max_length = max_length

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(
            str(row["text"]),
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "turn_uid":       row["turn_uid"],
            "is_split":       int(row["split"]),
        }
        for key in self.model_keys:
            item[f"label_{key}"] = LABEL2IDX.get(row[f"label_{key}"], LABEL2IDX["neutral"])
        return item

BATCH = 8
train_df = turns_wide[turns_wide["split_set"]=="train"]
val_df   = turns_wide[turns_wide["split_set"]=="val"]
test_df  = turns_wide[turns_wide["split_set"]=="test"]

train_loader = DataLoader(MultiHeadDataset(train_df, loaded), batch_size=BATCH,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(MultiHeadDataset(val_df,   loaded), batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MultiHeadDataset(test_df,  loaded), batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

In [ ]:
class MultiHeadStanceModel(nn.Module):
    def __init__(self, encoder_name, model_keys, num_labels=5):
        super().__init__()
        self.encoder = DebertaV2Model.from_pretrained(encoder_name)
        self.heads   = nn.ModuleDict({
            key: nn.Linear(768, num_labels) for key in model_keys
        })

    def forward(self, input_ids, attention_mask):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = enc.last_hidden_state[:, 0, :].float()
        logits = {key: head(cls) for key, head in self.heads.items()}
        return logits, cls

model = MultiHeadStanceModel(MODEL_NAME, loaded).to(device)
print(f"Encoder params: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"Heads: {list(model.heads.keys())}")

## Training

- Epoch 1: encoder frozen, heads only (lr=1e-3)
- Epochs 2–3: full fine-tune, encoder at lr=2e-5
- Gradient accumulation 4 steps → effective batch 32

In [ ]:
# Resume from checkpoint (SKIP_TRAINING=True to load, False to train)
SKIP_TRAINING   = False
CKPT_FROM_DRIVE = True

if SKIP_TRAINING:
    if CKPT_FROM_DRIVE:
        from google.colab import drive
        drive.mount("/drive", force_remount=False)
        CKPT_PATH = "/drive/MyDrive/central_bank_spillovers/multihead_turns/model_turns6_checkpoint.pt"
    else:
        from google.colab import files as colab_files
        print("Upload model_turns6_checkpoint.pt")
        up = colab_files.upload()
        CKPT_PATH = list(up.keys())[0]

    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)  # own checkpoint (trusted); needed since torch 2.6 defaults weights_only=True
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.float().to(device)
    history       = ckpt.get("history", [])
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print(f"Loaded: {CKPT_PATH}")
else:
    print("SKIP_TRAINING=False — run training cell next.")

In [ ]:
if not SKIP_TRAINING:
    EPOCHS = 3; GRAD_ACC = 4
    model  = model.float()
    crit   = nn.CrossEntropyLoss()

    def make_opt(m, enc_lr=2e-5, head_lr=1e-3):
        return torch.optim.AdamW([
            {"params": m.encoder.parameters(), "lr": enc_lr, "weight_decay": 0.01},
            {"params": m.heads.parameters(),   "lr": head_lr, "weight_decay": 0.01},
        ])

    @torch.no_grad()
    def evaluate(loader):
        model.eval()
        preds = {k:[] for k in loaded}; trues = {k:[] for k in loaded}; tot = 0.0
        for b in loader:
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            lg, _ = model(ids, mask)
            loss = sum(crit(lg[k], b[f"label_{k}"].to(device)) for k in loaded)
            tot += loss.item()
            for k in loaded:
                preds[k].extend(lg[k].argmax(-1).cpu().tolist())
                trues[k].extend(b[f"label_{k}"].tolist())
        accs = {k: np.mean(np.array(preds[k])==np.array(trues[k])) for k in loaded}
        return tot/len(loader), accs

    history = []; best_val_loss = float("inf")

    for epoch in range(1, EPOCHS+1):
        for p in model.encoder.parameters(): p.requires_grad = (epoch > 1)
        opt = make_opt(model, enc_lr=(2e-5 if epoch>1 else 0))
        print(f"Epoch {epoch}: {'frozen' if epoch==1 else 'full fine-tune'}")
        model.train(); opt.zero_grad(); run_loss = 0.0
        for step, b in enumerate(train_loader, 1):
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            lg, _ = model(ids, mask)
            loss  = sum(crit(lg[k], b[f"label_{k}"].to(device)) for k in loaded) / GRAD_ACC
            loss.backward(); run_loss += loss.item() * GRAD_ACC
            if step % GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            if step % 50 == 0:
                print(f"  step {step}/{len(train_loader)}  loss={run_loss/step:.4f}")
        vl, va = evaluate(val_loader)
        print(f"  val_loss={vl:.4f}  accs={va}")
        history.append({"epoch":epoch,"val_loss":vl,"val_accs":va})
        if vl < best_val_loss: best_val_loss = vl
    print("Training complete.")

In [ ]:
if not SKIP_TRAINING:
    from google.colab import drive as _drv
    _drv.mount("/drive", force_remount=False)
    DRIVE_DIR = "/drive/MyDrive/central_bank_spillovers/multihead_turns"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CKPT_PATH = os.path.join(DRIVE_DIR, "model_turns6_checkpoint.pt")
    torch.save({"model_state_dict":model.state_dict(),"loaded_keys":loaded,
                "label2idx":LABEL2IDX,"model_name":MODEL_NAME,
                "history":history,"best_val_loss":best_val_loss}, CKPT_PATH)
    print(f"Saved: {CKPT_PATH}")

## 5. Evaluation — Per-Head Metrics

In [ ]:
model.eval()
all_preds = {k:[] for k in loaded}; all_true = {k:[] for k in loaded}

with torch.no_grad():
    for b in test_loader:
        ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
        lg, _ = model(ids, mask)
        for k in loaded:
            all_preds[k].extend(lg[k].argmax(-1).cpu().tolist())
            all_true[k].extend(b[f"label_{k}"].tolist())

rows = []
print("=== Test Set ===")
for k in loaded:
    yp = np.array(all_preds[k]); yt = np.array(all_true[k])
    acc = accuracy_score(yt,yp); f1 = f1_score(yt,yp,average="macro",zero_division=0)
    rows.append({"Model":MODELS[k],"Accuracy":f"{acc:.3f}","Macro-F1":f"{f1:.3f}"})
    print(f"{MODELS[k]:18s}  acc={acc:.3f}  macro-F1={f1:.3f}")

pd.DataFrame(rows).to_csv("per_head_metrics_turns.csv",index=False)
display(pd.DataFrame(rows))

In [ ]:
ncol = 3
nrow = int(np.ceil(len(loaded)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 4*nrow))
axes = np.array(axes).reshape(-1)
lnames = list(LABEL2IDX.keys())
for ax,k in zip(axes,loaded):
    cm = confusion_matrix(all_true[k],all_preds[k], labels=list(range(5)))
    ConfusionMatrixDisplay(cm,display_labels=lnames).plot(ax=ax,colorbar=False,xticks_rotation=45)
    ax.set_title(MODELS[k])
for ax in axes[len(loaded):]:
    ax.axis("off")
plt.suptitle("Confusion Matrices — Multi-Head DeBERTa (Turns, 6 models)",y=1.01)
plt.tight_layout()
plt.savefig("confusion_matrices_turns.png",dpi=120,bbox_inches="tight")
plt.show()

## 6. Linear Probe + Base-vs-Finetuned

Train logistic regression on the **fine-tuned** DeBERTa CLS vectors to predict `split`, then
repeat on CLS vectors from a **frozen pretrained** DeBERTa (no fine-tuning). The gap quantifies
what fine-tuning on the six label sets adds over raw pretrained semantics.

**Key table:** Dictionary → TF-IDF → frozen DeBERTa → fine-tuned DeBERTa AUC.

The companion cell then checks whether the same representation predicts the **graded**
disagreement targets (`label_entropy`, `score_std`), and whether the binary-split risk score
ranks turns the same way as cross-sectional score dispersion — the Malta synthetic-traders
disagreement measure.

In [ ]:
@torch.no_grad()
def extract_cls(encoder, df, max_length=512, batch_size=16):
    """Mean-free CLS extraction from any DebertaV2Model encoder."""
    ds  = MultiHeadDataset(df.reset_index(drop=True), loaded, max_length=max_length)
    ldr = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    out = []
    encoder.eval()
    for b in ldr:
        ids=b["input_ids"].to(device); mask=b["attention_mask"].to(device)
        enc = encoder(input_ids=ids, attention_mask=mask)
        out.append(enc.last_hidden_state[:,0,:].float().cpu().numpy())
    return np.vstack(out)

# Fine-tuned encoder CLS
X_ft = extract_cls(model.encoder, turns_wide)
# Frozen pretrained encoder CLS (fresh download, never fine-tuned)
frozen_encoder = DebertaV2Model.from_pretrained(MODEL_NAME).to(device)
X_base = extract_cls(frozen_encoder, turns_wide)

y_all      = turns_wide["split"].astype(int).values
split_sets = turns_wide["split_set"].values
print(f"CLS matrices: fine-tuned {X_ft.shape}, frozen {X_base.shape}")

In [ ]:
tr_m = split_sets=="train"
te_m = split_sets=="test"

def probe_auc(X):
    sc = StandardScaler()
    Xtr = sc.fit_transform(X[tr_m]); Xte = sc.transform(X[te_m])
    clf = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced")
    clf.fit(Xtr, y_all[tr_m])
    auc = roc_auc_score(y_all[te_m], clf.predict_proba(Xte)[:,1])
    return auc, sc, clf

auc_base, _, _              = probe_auc(X_base)
auc_ft, sc_ft, probe_ft    = probe_auc(X_ft)

print("=== AUC Comparison (predicting binary split) ===")
print(f"  Dictionary (hawk/dove count) : {auc_dict:.3f}")
print(f"  TF-IDF logistic regression   : {auc_tfidf:.3f}")
print(f"  Frozen pretrained DeBERTa    : {auc_base:.3f}")
print(f"  Fine-tuned DeBERTa CLS       : {auc_ft:.3f}")
print(f"\n  Fine-tuning lift over frozen : {auc_ft-auc_base:+.3f}")

pd.DataFrame([
    {"method":"Dictionary","auc":round(auc_dict,3)},
    {"method":"TF-IDF","auc":round(auc_tfidf,3)},
    {"method":"Frozen DeBERTa","auc":round(auc_base,3)},
    {"method":"Fine-tuned DeBERTa","auc":round(auc_ft,3)},
]).to_csv("probe_auc_comparison_turns.csv", index=False)

In [ ]:
# Continuous companions: how well does the fine-tuned CLS predict graded disagreement?
ent = turns_wide["label_entropy"].values
gap = turns_wide["max_ordinal_gap"].values
std = turns_wide["score_std"].values

sc_e = StandardScaler()
Xtr_e = sc_e.fit_transform(X_ft[tr_m]); Xte_e = sc_e.transform(X_ft[te_m])

# (a) nominal disagreement — entropy
reg_ent = Ridge(alpha=1.0).fit(Xtr_e, ent[tr_m])
ent_pred_te = reg_ent.predict(Xte_e)
r2_ent  = r2_score(ent[te_m], ent_pred_te)
rho_ent = spearmanr(ent[te_m], ent_pred_te).correlation

# (b) ordinal dispersion — cross-sectional SD (Malta synthetic-traders analogue)
reg_std = Ridge(alpha=1.0).fit(Xtr_e, std[tr_m])
std_pred_te = reg_std.predict(Xte_e)
r2_std  = r2_score(std[te_m], std_pred_te)
rho_std = spearmanr(std[te_m], std_pred_te).correlation

# Risk score from the split probe, all turns
disagree_risk = probe_ft.predict_proba(sc_ft.transform(X_ft))[:,1]
rho_risk_ent = spearmanr(disagree_risk, ent).correlation
rho_risk_gap = spearmanr(disagree_risk, gap).correlation
rho_risk_std = spearmanr(disagree_risk, std).correlation

print("Continuous companions (fine-tuned CLS):")
print(f"  Ridge R^2 predicting label_entropy (test) : {r2_ent:.3f}  (Spearman {rho_ent:.3f})")
print(f"  Ridge R^2 predicting score_std    (test) : {r2_std:.3f}  (Spearman {rho_std:.3f})")
print("\nDoes the binary-split risk score also track graded disagreement?")
print(f"  Spearman(disagree_risk, label_entropy)    : {rho_risk_ent:.3f}")
print(f"  Spearman(disagree_risk, max_ordinal_gap)  : {rho_risk_gap:.3f}")
print(f"  Spearman(disagree_risk, score_std)        : {rho_risk_std:.3f}")

In [ ]:
all_df = turns_wide.reset_index(drop=True).copy()
all_df["disagree_risk"] = disagree_risk
all_df["date"] = pd.to_datetime(all_df["date"].astype(str), format="%Y%m%d", errors="coerce")
# BoE dates are YYYYMM; retry those that failed 8-digit parse
mask_na = all_df["date"].isna()
all_df.loc[mask_na,"date"] = pd.to_datetime(
    turns_wide.reset_index(drop=True).loc[mask_na,"date"].astype(str), format="%Y%m", errors="coerce")
all_df["split_int"] = all_df["split"].astype(int)

# Meeting-level aggregation of every disagreement measure
meeting_risk = (all_df.groupby(["bank","date"])
                .agg(disagree_risk=("disagree_risk","mean"),     # probe P(split)
                     split_rate=("split_int","mean"),            # raw binary disagreement rate
                     label_entropy=("label_entropy","mean"),     # nominal entropy
                     score_std=("score_std","mean"),             # ordinal dispersion (stdev)
                     mean_score=("mean_score","mean"))           # stance signal (not disagreement)
                .reset_index().sort_values(["bank","date"]))

# Min-max normalise each disagreement measure to [0,1] so they overlay on one axis
MEASURES = {
    "disagree_risk": ("probe risk P(split)",   "#C44E52"),
    "split_rate":    ("binary split rate",     "#7f7f7f"),
    "label_entropy": ("label entropy",         "#2ca02c"),
    "score_std":     ("score SD (dispersion)", "#4C72B0"),
}
for col in MEASURES:
    lo, hi = meeting_risk[col].min(), meeting_risk[col].max()
    meeting_risk[f"{col}_n"] = (meeting_risk[col] - lo) / (hi - lo) if hi > lo else 0.0

banks = sorted(meeting_risk["bank"].unique())
fig, axes = plt.subplots(len(banks), 1, figsize=(14, 3.2*len(banks)), sharex=True)
if len(banks) == 1: axes = [axes]
for ax, bank in zip(axes, banks):
    bdf = meeting_risk[meeting_risk["bank"] == bank]
    for col, (lbl, color) in MEASURES.items():
        ax.plot(bdf["date"], bdf[f"{col}_n"], color=color, linewidth=1.2, alpha=0.85, label=lbl)
    ax.fill_between(bdf["date"], bdf["disagree_risk_n"], alpha=0.12, color="#C44E52")
    ax.set_ylim(0, 1); ax.set_ylabel("normalised [0,1]"); ax.set_title(bank)
axes[0].legend(ncol=4, fontsize=8, loc="upper right")
plt.xlabel("Date")
plt.suptitle("Disagreement over time — four measures (min-max normalised), 6 models", fontsize=11)
plt.tight_layout()
plt.savefig("linear_probe_risk_timeline_turns.png", dpi=130, bbox_inches="tight")
plt.show()

# Do the measures agree on WHEN disagreement is high?
print("Meeting-level Spearman correlations between disagreement measures:")
print(meeting_risk[list(MEASURES)].corr(method="spearman").round(2).to_string())

all_df[["turn_uid","bank","date","split","label_entropy","max_ordinal_gap",
        "mean_score","score_std","mean_pair_d","disagree_risk"]].to_csv(
    "linear_probe_scores_turns.csv", index=False)
meeting_risk.to_csv("linear_probe_meeting_risk_turns.csv", index=False)
print("\nSaved probe scores + meeting risk (probe / binary / entropy / score_std).")

## 7. Swap Analysis — Encoding vs. Decision-Rule Disagreement

Run the encoder once per split turn; apply all 6 heads to the **identical** CLS vector.
- If heads agree: zero-shot disagreement was encoding-level (different LLM architectures)
- If heads still disagree: genuine decision-rule difference

With 6 models, unanimous = all 6 heads predict the same label.

In [ ]:
from itertools import combinations

test_split_df = turns_wide[(turns_wide["split_set"]=="test") & (turns_wide["split"])].reset_index(drop=True)
split_loader  = DataLoader(MultiHeadDataset(test_split_df,loaded,max_length=512),
                           batch_size=4,shuffle=False)
pairs = list(combinations(loaded,2))

model.eval()
swap_results = []
pneu = {k:[] for k in loaded}
with torch.no_grad():
    for batch in split_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        _,cls = model(ids,mask)
        swap_preds = {k: model.heads[k](cls).argmax(-1).cpu().tolist() for k in loaded}
        probs      = {k: torch.softmax(model.heads[k](cls),-1)[:,LABEL2IDX["neutral"]].cpu().tolist() for k in loaded}
        for i in range(cls.shape[0]):
            labels_i = [swap_preds[k][i] for k in loaded]
            swap_results.append({"n_unique":len(set(labels_i)),
                                 **{f"pred_{k}":swap_preds[k][i] for k in loaded}})
            for k in loaded: pneu[k].append(probs[k][i])

swap_df = pd.DataFrame(swap_results)
total   = len(swap_df)
print(f"Swap Analysis — Consensus on {total} Split Turns (Shared Encoder):")
for n_u,cnt in swap_df["n_unique"].value_counts().sort_index().items():
    tag = "all agree" if n_u==1 else ("all differ" if n_u==len(loaded) else "partial")
    print(f"  {n_u} unique predictions ({tag}): {cnt} ({cnt/total:.1%})")

unan = (swap_df["n_unique"]==1).mean()
print(f"\n{unan:.1%} resolve to unanimous agreement when encoding is shared (encoding-level)")
print(f"{1-unan:.1%} are genuine decision-rule disagreements")

print("\nPairwise disagreement rates (shared CLS):")
pair_rows = []
for a,b in pairs:
    d = (swap_df[f"pred_{a}"] != swap_df[f"pred_{b}"]).mean()
    pair_rows.append((MODELS[a],MODELS[b],d))
for a,b,d in sorted(pair_rows,key=lambda x:x[2]):
    print(f"  {a:18s} <-> {b:18s}: {d:.1%}")

print("\nMean P(neutral) per head on split turns (shared CLS):")
for k in sorted(loaded, key=lambda k: -np.mean(pneu[k])):
    print(f"  {MODELS[k]:18s} {np.mean(pneu[k]):.2f}")

swap_df.to_csv("swap_analysis_turns.csv",index=False)

## 8. Gradient Attribution per Head

For each split turn in the test set: gradient of the predicted-class logit w.r.t. input word
embeddings (∂logit/∂embedding), L2-normed per token. *Turns truncated to 512 tokens.*

In [ ]:
STOP_WORDS = {
    "the","and","of","to","in","is","that","for","it","we","on","at","a","an",
    "be","as","by","are","this","with","have","from","our","has","was",
    "will","been","or","which","but","not","##","[CLS]","[SEP]",
}

token_scores = {k:{} for k in loaded}
model.eval()
for batch in split_loader:
    ids  = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    for k in loaded:
        emb = model.encoder.embeddings.word_embeddings(ids)
        emb.retain_grad()
        out = model.encoder(inputs_embeds=emb, attention_mask=mask)
        cls = out.last_hidden_state[:,0,:].float()
        lg  = model.heads[k](cls)
        lg[range(len(lg)),lg.argmax(-1)].sum().backward(retain_graph=True)
        if emb.grad is None: model.zero_grad(); continue
        imp = emb.grad.norm(dim=-1).detach().cpu().numpy()
        for b in range(ids.shape[0]):
            toks = tokenizer.convert_ids_to_tokens(ids[b].cpu().tolist())
            for tok,sc in zip(toks,imp[b]):
                clean = tok.replace("\u2581","").replace("##","").lower()
                if clean and clean not in STOP_WORDS and len(clean)>2:
                    token_scores[k][clean] = token_scores[k].get(clean,0)+sc
        model.zero_grad()

print(f"Attribution done over {len(test_split_df)} split turns.")

In [ ]:
TOP_N = 20
ncol  = 3
nrow  = int(np.ceil(len(loaded)/ncol))
palette = sns.color_palette("tab10", len(loaded))
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol,6*nrow))
axes = np.array(axes).reshape(-1)
for ax,k,col in zip(axes,loaded,palette):
    items = sorted(token_scores[k].items(),key=lambda x:x[1],reverse=True)[:TOP_N]
    toks,vals = zip(*items)
    ax.barh(range(TOP_N),list(vals)[::-1],color=col,alpha=0.85)
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(list(toks)[::-1],fontsize=8)
    ax.set_title(MODELS[k])
    ax.set_xlabel("Cumulative gradient norm")
for ax in axes[len(loaded):]:
    ax.axis("off")
plt.suptitle("Gradient Attribution per Head (Split Turns, 6 models)",y=1.005)
plt.tight_layout()
plt.savefig("head_gradient_attribution_turns.png",dpi=130,bbox_inches="tight")
plt.show()

## 9. Head Weight Geometry

Each head is `Linear(768,5)`. Cosine similarity between flattened 5×768 weight matrices
measures how similar two heads' learned decision rules are (15 pairs for 6 models).

In [ ]:
from sklearn.decomposition import PCA

hw = {k: model.heads[k].weight.detach().cpu().numpy().flatten() for k in loaded}

# Cosine similarity matrix
M = np.zeros((len(loaded),len(loaded)))
for i,a in enumerate(loaded):
    for j,b in enumerate(loaded):
        M[i,j] = np.dot(hw[a],hw[b])/(np.linalg.norm(hw[a])*np.linalg.norm(hw[b]))

print("Head weight cosine similarities (sorted):")
for a,b in sorted(pairs, key=lambda p: -M[loaded.index(p[0]),loaded.index(p[1])]):
    cos = M[loaded.index(a),loaded.index(b)]
    print(f"  {MODELS[a]:18s} <-> {MODELS[b]:18s}: {cos:.3f}")

fig, axes = plt.subplots(1,2,figsize=(13,5))
sns.heatmap(M, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=[MODELS[k] for k in loaded],
            yticklabels=[MODELS[k] for k in loaded], ax=axes[0])
axes[0].set_title("Head Weight Cosine Similarity")
plt.setp(axes[0].get_xticklabels(), rotation=40, ha="right")

W      = np.stack([hw[k] for k in loaded])
pca2   = PCA(n_components=2).fit(W)
coords = pca2.transform(W)
palette = sns.color_palette("tab10", len(loaded))
for i,(k,col) in enumerate(zip(loaded,palette)):
    axes[1].scatter(*coords[i],s=160,color=col,zorder=3)
    axes[1].annotate(MODELS[k],coords[i],textcoords="offset points",xytext=(8,4),fontsize=9)
axes[1].set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.0%} var)")
axes[1].set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.0%} var)")
axes[1].set_title("PCA of Head Weight Matrices")
plt.tight_layout()
plt.savefig("head_weight_similarity_turns.png",dpi=130,bbox_inches="tight")
plt.show()

## 10. Save & Download All Outputs

In [ ]:
import shutil

outputs = [
    "per_head_metrics_turns.csv",
    "confusion_matrices_turns.png",
    "tfidf_shap_disagreement_turns.png",
    "probe_auc_comparison_turns.csv",
    "linear_probe_risk_timeline_turns.png",
    "linear_probe_scores_turns.csv",
    "linear_probe_meeting_risk_turns.csv",
    "swap_analysis_turns.csv",
    "head_gradient_attribution_turns.png",
    "head_weight_similarity_turns.png",
]

if os.path.exists("/content"):
    if not os.path.exists("/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/drive",force_remount=False)
    DRIVE_DIR = "/drive/MyDrive/central_bank_spillovers/multihead_turns"
    os.makedirs(DRIVE_DIR,exist_ok=True)
    for fname in outputs:
        if os.path.exists(fname):
            shutil.copy(fname,os.path.join(DRIVE_DIR,fname))
            print(f"  saved: {fname}")
        else:
            print(f"  MISSING: {fname}")
    from google.colab import files
    for fname in outputs:
        if os.path.exists(fname): files.download(fname)
else:
    print("Local run — outputs written to working directory:")
    for fname in outputs:
        print(f"  {'OK ' if os.path.exists(fname) else 'MISS'} {fname}")